# Regression Methods Final Project

In [ ]:
import csv
import numpy as np
from math import sqrt
from numpy import linalg as LA
from sklearn.impute import SimpleImputer, KNNImputer
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression

Linear Regression:

In [71]:
class LinearRegression1:
    def __init__(self):
        self.weights = None
        self.bias = None

    def fit(self, X, y):
        n_features = X.shape[1]
        self.bias = 0
        self.weights = np.zeros(n_features)
        X = np.concatenate((np.ones((X.shape[0], 1)), X), axis=1)
        X = X.astype('float64')
        y = y.astype('float64')
        coefficients = np.linalg.inv(X.T.dot(X)).dot(X.T).dot(y)
        self.bias = coefficients[0]
        self.weights = coefficients[1:]

    def predict(self, X):
        y_predicted = np.dot(X, self.weights) + self.bias
        return y_predicted


Logistic Regression:

In [72]:
class My_LogisticRegression:
    def __init__(self, learning_rate=0.25, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.weights, self.bias = None, None

    @staticmethod
    def _sigmoid(x):
        return 1 / (1 + np.exp(-x))

    @staticmethod
    def _binary_cross_entropy(y, y_hat):

        def safe_log(x):
            return 0 if x == 0 else np.log(x)

        total = 0
        for curr_y, curr_y_hat in zip(y, y_hat):
            total += (curr_y * safe_log(curr_y_hat) + (1 - curr_y) * safe_log(1 - curr_y_hat))
        return - total / len(y)

    def fit(self, X, y):
        X = X.astype(np.float64)
        self.weights = np.zeros(X.shape[1])
        self.bias = 0

        for i in range(self.n_iterations):
            linear_pred = np.dot(X, self.weights) + self.bias
            probability = self._sigmoid(linear_pred)
            # Calculate derivatives
            partial_w = (1 / X.shape[0]) * (np.dot(X.T, (probability - y)))
            partial_d = (1 / X.shape[0]) * (np.sum(probability - y))

            self.weights -= self.learning_rate * partial_w
            self.bias -= self.learning_rate * partial_d

    def predict_proba(self, X):
        X = X.astype(np.float64)
        linear_pred = np.dot(X, self.weights) + self.bias
        return self._sigmoid(linear_pred)

    def predict(self, X, threshold=0.5):
        X = X.astype(np.float64)
        probabilities = self.predict_proba(X)
        return [1 if i > threshold else 0 for i in probabilities]


PCA:

In [73]:
class MY_PCA:
    def __init__(self, k):
        self.k = k
        self.normalized_matrix = None
        self.cov = None
        self.values = None
        self.vectors = None

    def standartization(self, data):
        mean = np.mean(data, axis=0)
        std = np.std(data, axis=0,ddof=1)
        self.normalized_matrix = (data-mean) / std

    def cov_var_matrix(self):
        cov_mat = np.cov(self.normalized_matrix, rowvar=False, bias=False)
        self.cov = cov_mat

    def eigen(self):
        w, v = LA.eig(self.cov)
        idx = w.argsort()[::-1]
        w = w[idx]
        v = v[:, idx]
        selected_w = w.T[:self.k]
        selected_V = v.T[:self.k]
        self.values = selected_w
        self.vectors = selected_V

    def fit(self, data):
        self.standartization(data)
        self.cov_var_matrix()
        self.eigen()

    def transformation(self):
        new_X = np.matmul(self.normalized_matrix, self.vectors.T)
        return new_X



Min-Max on train data

In [76]:
def minmax_on_train(data):
    min_max_values = []
    data_without_first_row = data[1:]
    for i, column in enumerate(data_without_first_row.T):
        min_value = np.min(column)
        max_value = np.max(column)
        min_max_values.append((min_value, max_value))
        for j, value in enumerate(column):
            scaled_value = (value - min_value) / (max_value - min_value)
            data_without_first_row[j, i] = scaled_value
    data_after_min_max = np.vstack((data[0], data_without_first_row))
    return data_after_min_max, min_max_values



Min-Max on test data:

In [77]:

def min_max_on_test(data, min_max_by_train):
    test_min_max = data[1:]
    for i, column in enumerate(test_min_max.T):
        min_value = np.min(min_max_by_train[i][0])
        max_value = np.max(min_max_by_train[i][1])
        for j, value in enumerate(column):
            scaled_value = (value - min_value) / (max_value - min_value)
            test_min_max[j, i] = scaled_value
    test_min_max = np.vstack((data[0], test_min_max))
    return test_min_max


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression

# load data
data = load_breast_cancer()
X = data.data
y = data.target

# split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# normalize
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# YOUR model
my_model = My_LogisticRegression()
my_model.fit(X_train, y_train)
y_pred = my_model.predict(X_test)

# sklearn model
sk_model = LogisticRegression(max_iter=1000)
sk_model.fit(X_train, y_train)
y_pred_sk = sk_model.predict(X_test)

# results
print("My Logistic Accuracy:", accuracy_score(y_test, y_pred))
print("Sklearn Accuracy:", accuracy_score(y_test, y_pred_sk))

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression

# load data
data = load_diabetes()
X = data.data
y = data.target

# split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# normalize
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# YOUR model
my_model = LinearRegression1()
my_model.fit(X_train, y_train)
y_pred = my_model.predict(X_test)

# sklearn model
sk_model = LinearRegression()
sk_model.fit(X_train, y_train)
y_pred_sk = sk_model.predict(X_test)

# results
print("My MSE:", mean_squared_error(y_test, y_pred))
print("Sklearn MSE:", mean_squared_error(y_test, y_pred_sk))

In [ ]:
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA

# load data
data = load_iris()
X = data.data

# YOUR PCA
my_pca = MY_PCA(2)
my_pca.fit(X)
X_reduced = my_pca.transformation()

# sklearn PCA
sk_pca = PCA(n_components=2)
X_reduced_sk = sk_pca.fit_transform(X)

print("My PCA shape:", X_reduced.shape)
print("Sklearn PCA shape:", X_reduced_sk.shape)